# Week 7 Exercise: Train Model to Predict Product Prices

In [ ]:
!pip install -q trl bitsandbytes==0.46.1

## Import Required Libraries

In [2]:
import os
import torch
import re
from dotenv import load_dotenv
from huggingface_hub import login
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from google.colab import userdata

## Login to HuggingFace

In [3]:
load_dotenv()
hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

## Load Dataset

In [ ]:
dataset = load_dataset("ed-donner/items_lite")
train = dataset['train'].select(range(1000))
val = dataset['validation']
test = dataset['test']

print(len(train), len(val), len(test))

## Prompt Formatting Function

In [ ]:
def build_text(example):
    return {
        "text":
            "You are a retail price estimator.\n\n"
            f"Product: {example['title']}\n"
            f"Description: {example['summary']}\n\n"
            f"Price: {example['price']}"
    }


train = train.map(build_text)
val = val.map(build_text)

# # Remove the 'prompt' column to prevent SFTTrainer from getting confused
train = train.remove_columns("prompt")
val = val.remove_columns("prompt")

In [ ]:
capability = torch.cuda.get_device_capability()
use_bf16 = capability[0] >= 8

print(torch.cuda.is_available())
print(torch.cuda.get_device_name())

## Quantize to 4 bits

In [7]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
)

## Load Base Model

In [ ]:
BASE_MODEL = 'Qwen/Qwen2.5-3B'

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
)

model.generation_config.pad_token_id = tokenizer.pad_token_id

## Configure LoRA for Efficient Fine-Tuning

In [9]:
lora_config = LoraConfig(
    r=32,
    lora_alpha=32 * 2,
    lora_dropout=0.1,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
)

model = get_peft_model(model, lora_config)

## Training Arguments

In [ ]:
sft_config = SFTConfig(
    output_dir="./price_model",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    dataset_text_field="text",
    max_length=96,
    packing=True,
    logging_steps=10,
    optim="paged_adamw_32bit",
    weight_decay=0.001,
    bf16=True,
    max_grad_norm=0.3,
    warmup_ratio=0.01,
    lr_scheduler_type="cosine",
    eval_strategy="no",
    save_strategy="epoch",
    report_to="none"
)

## Train the Model

In [ ]:
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train,
    eval_dataset=val,
    processing_class=tokenizer,
)

trainer.train()

## Build Prediction Prompt

In [12]:
def build_prompt(item):
    return (
        'You are a retail price estimator.\n\n'
        f"Product: {item['title']}\n"
        f"Description: {item['summary']}\n\n"
        'Price:'
    )

## Prediction Function

In [13]:
def predict_price(item):
    prompt = build_prompt(item)

    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=8,
        do_sample=False
    )

    text = tokenizer.decode(outputs[0])
    match = re.findall(r'\d+\.?\d*', text)

    return float(match[0]) if match else 0

## Evaluate Model

In [14]:
def evaluate(dataset):
    errors = []

    for item in dataset:
        pred = predict_price(item)
        actual = item['price']

        errors.append(abs(pred - actual))

    return sum(errors) / len(errors)

In [ ]:
error = evaluate(test)
print('Average prediction error:', error)